# 01 - Generate Synthetic Airport Operations Data (Bronze)

Generates deterministic, privacy-safe airport operations data for the Airport Operations Intelligence demonstration.

## Public references and synthetic operations

Airport, country, airline, manufacturer, and aircraft-type identity attributes are public references. All ownership, concessions, facilities, fleets, registrations, routes, schedules, flights, passengers, employees, operations, transactions, incidents, performance, recommendations, and outcomes are synthetic.

The default catalog contains 18 sourced airport anchors across France, Italy, Portugal, and Jordan, 20 requested public airline references, and 16 public aircraft-type references. Seat capacity and turnaround targets are synthetic operating assumptions. The fictional organization does not own or operate the referenced airports.

The notebook writes source-shaped `bronze_*` Delta tables to the default `AirportOpsLakehouse`. Incremental mode uses deterministic Delta MERGE keys. Recommendations remain advisory and require human review.

In [ ]:
# PARAMETERS - override artifact_root/profile through the notebook job API when needed.
import json
from pathlib import Path

artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
simulation_profile_override = ''
config_path = Path(artifact_root) / 'config' / 'demo_config.json'
reference_root = Path(artifact_root) / 'data' / 'reference'

demo_config = json.loads(config_path.read_text(encoding='utf-8'))
profiles = json.loads((Path(artifact_root) / demo_config['simulation_profiles_path']).read_text(encoding='utf-8'))['profiles']
selected_profile = simulation_profile_override or demo_config['simulation_profile']
assert selected_profile in profiles, 'Unknown simulation profile: ' + selected_profile
for profile_key, profile_value in profiles[selected_profile].items():
    if profile_key != 'description':
        demo_config[profile_key] = profile_value
demo_config['simulation_profile'] = selected_profile

num_airports = int(demo_config['airport_count'])
num_airlines = int(demo_config['airline_count'])
num_aircraft_types = int(demo_config['aircraft_type_count'])
gates_per_airport = int(demo_config['gates_per_airport'])
sim_hours = int(demo_config['simulation_hours'])
random_seed = int(demo_config['random_seed'])
base_date = demo_config['base_date']
flights_per_gate = int(demo_config['flights_per_gate'])
queue_interval_minutes = int(demo_config['queue_interval_minutes'])

assert num_airports == 18 and num_airlines == 20 and num_aircraft_types == 16
assert random_seed == 42
assert int(demo_config['operating_region_count']) == 4
assert int(demo_config['scale_factor']) >= 1
assert gates_per_airport >= 1 and sim_hours >= 1
assert queue_interval_minutes > 0 and (sim_hours * 60) % queue_interval_minutes == 0
assert demo_config['deployment']['dry_run'] is True
print('Simulation profile:', selected_profile, profiles[selected_profile]['description'])

In [ ]:
import random
from datetime import datetime, timedelta

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, Row, functions as F

random.seed(random_seed)
BASE = datetime.strptime(base_date, '%Y-%m-%d')
BATCH_ID = f"AIRPORT-OPS-{base_date.replace('-', '')}-SEED-{random_seed}-SCALE-{demo_config['scale_factor']}"
PROCESSING_MODE = demo_config.get('processing_mode', 'full_reset')
assert PROCESSING_MODE in {'full_reset', 'incremental'}

energy_sources = ['HVAC', 'Lighting', 'Jetbridge', 'GPU']
checkpoints = ['CheckIn', 'Security', 'Immigration', 'Boarding']
delay_reasons = ['Weather', 'Technical', 'FlowControl', 'Crew', 'LateInbound', 'Baggage']
PRIMARY_KEYS = {
    'bronze_demo_config':'config_id','bronze_country':'country_id','bronze_airport':'airport_id',
    'bronze_aircraft':'aircraft_type_id','bronze_airline':'airline_id',
    'bronze_corporate_headquarters':'headquarters_id','bronze_operating_region':'operating_region_id',
    'bronze_airport_group_assignment':'group_assignment_id','bronze_runway_reference':'runway_reference_id',
    'bronze_airline_aircraft_eligibility':'eligibility_id','bronze_airline_airport_service':'service_id',
    'bronze_gate':'gate_id','bronze_service_team':'team_id','bronze_flight_turnaround':'flight_event_id',
    'bronze_passenger_queue':'queue_metric_id','bronze_energy':'meter_reading_id',
    'bronze_maintenance':'maintenance_id','bronze_weather':'weather_id',
    'bronze_operational_incidents':'incident_id'}


def ts(hour, minute=0):
    return BASE + timedelta(hours=hour, minutes=minute)


def load_catalog(file_name):
    payload = json.loads((reference_root / file_name).read_text(encoding='utf-8'))
    records = []
    for source_record in payload['records']:
        record = dict(source_record)
        record.setdefault('data_classification', 'PublicReference')
        record.setdefault('source_as_of_date', payload['source_as_of_date'])
        record.setdefault('source_name', 'CheckedInPublicReferenceCatalog')
        record.setdefault('source_url', 'repo://data/reference/' + file_name)
        record.setdefault('record_source', 'CheckedInPublicReferenceCatalog')
        record.setdefault('is_synthetic', False)
        records.append(record)
    return records


def write_delta(rows_or_frame, name, data_classification=None):
    if isinstance(rows_or_frame, DataFrame):
        frame = rows_or_frame
    else:
        if not rows_or_frame:
            raise ValueError(f'Cannot write empty row collection to {name}')
        frame = spark.createDataFrame(rows_or_frame)
    public_reference_tables = {'bronze_airport','bronze_airline','bronze_aircraft','bronze_country'}
    synthetic_master_tables = {
        'bronze_demo_config','bronze_gate','bronze_service_team','bronze_corporate_headquarters',
        'bronze_operating_region','bronze_airport_group_assignment','bronze_runway_reference',
        'bronze_airline_aircraft_eligibility','bronze_airline_airport_service'}
    default_classification = (
        'PublicReference' if name in public_reference_tables
        else 'SyntheticMaster' if name in synthetic_master_tables
        else 'SyntheticOperational')
    defaults = {
        'is_synthetic':F.lit(name not in public_reference_tables),
        'data_classification':F.lit(data_classification or default_classification),
        'source_name':F.lit('CheckedInPublicReferenceCatalog' if name in public_reference_tables else 'DeterministicSyntheticGenerator'),
        'source_url':F.lit('repo://data/reference/' if name in public_reference_tables else 'repo://notebooks/01_Generate_Sample_Data'),
        'source_as_of_date':F.lit(demo_config['reference_as_of_date'] if name in public_reference_tables else base_date),
        'generated_at_utc':F.current_timestamp(),'generator_version':F.lit(demo_config['generator_version']),
        'random_seed':F.lit(random_seed),'batch_id':F.lit(BATCH_ID),
        'record_source':F.lit('CheckedInPublicReferenceCatalog' if name in public_reference_tables else 'SyntheticGenerator')}
    for column_name, expression in defaults.items():
        if column_name not in frame.columns:
            frame = frame.withColumn(column_name, expression)

    if PROCESSING_MODE == 'incremental' and spark.catalog.tableExists(name):
        spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')
        key = PRIMARY_KEYS[name]
        (DeltaTable.forName(spark, name).alias('target')
            .merge(frame.alias('source'), f'target.{key} = source.{key}')
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        action = 'merged'
    else:
        frame.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable(name)
        action = 'overwrote'
    print(f'{action} {name}: {frame.count():,} source rows')

In [ ]:
# Public airport and country reference data with a fictional portfolio relationship.
airports = load_catalog('airports.json')
countries = load_catalog('countries.json')
assert len(airports) == num_airports
assert {airport['country'] for airport in airports} == {'France', 'Italy', 'Portugal', 'Jordan'}

airport_rows = [Row(
    airport_id=airport['airport_id'], airport_reference_id=airport['airport_reference_id'],
    iata_code=airport['iata_code'], icao_code=airport['icao_code'],
    airport_name=airport['airport_name'], city=airport['city'], country=airport['country'],
    region=airport['country'], country_code=airport['iso_country_code'],
    iso_country_code=airport['iso_country_code'], timezone=airport['iana_time_zone'],
    iana_time_zone=airport['iana_time_zone'], latitude=float(airport['latitude']),
    longitude=float(airport['longitude']), elevation_ft=int(airport['elevation_ft']),
    is_synthetic=False, reference_anchor_only=True, fictional_portfolio_relationship=True,
    data_classification='PublicReference', source_name=airport['source_name'],
    source_url=airport['source_url'], source_as_of_date=airport['source_as_of_date'],
    record_source=airport['record_source'])
    for airport in airports]

country_rows = [Row(
    country_id=country['country_id'], country_name=country['country_name'],
    iso_country_code=country['iso_country_code'], operating_region_id=country['operating_region_id'],
    iana_time_zones=country['iana_time_zones'], is_synthetic=False,
    data_classification='PublicReference', source_name=country['source_name'],
    source_url=country['source_url'], source_as_of_date=country['source_as_of_date'],
    record_source=country['record_source'])
    for country in countries]

expected_region_counts = {'France': 6, 'Italy': 5, 'Portugal': 5, 'Jordan': 2}
assert {region: sum(airport['country'] == region for airport in airports) for region in expected_region_counts} == expected_region_counts
assert len({airport['iata_code'] for airport in airports}) == 18
assert len({airport['icao_code'] for airport in airports}) == 18
assert all(not airport['is_synthetic'] and airport['reference_anchor_only'] for airport in airports)
assert all(airport['airport_id'].startswith('SYN-AP-') for airport in airports)

In [ ]:
# Public aircraft-type references with synthetic seat and turnaround assumptions.
aircraft = load_catalog('aircraft_types.json')
assert len(aircraft) == num_aircraft_types
assert len({item['icao_type_designator'] for item in aircraft}) == num_aircraft_types

aircraft_rows = [Row(
    aircraft_type_id=item['aircraft_type_id'], icao_type_code=item['icao_type_designator'],
    icao_type_designator=item['icao_type_designator'], model=item['model'],
    manufacturer=item['manufacturer'], category=item['aircraft_category'],
    aircraft_category=item['aircraft_category'], seats=int(item['representative_seat_capacity']),
    representative_seat_capacity=int(item['representative_seat_capacity']),
    turnaround_target_min=int(item['representative_turnaround_target_min']),
    representative_turnaround_target_min=int(item['representative_turnaround_target_min']),
    wingspan_m=float(item['wingspan_m']), length_m=float(item['length_m']),
    stand_category=item['stand_category'], operating_assumption_flag=True,
    operating_assumption_classification='SyntheticMaster', is_synthetic=False,
    data_classification='PublicReference', source_name=item['source_name'],
    source_url=item['source_url'], manufacturer_source_url=item['manufacturer_source_url'],
    source_as_of_date=item['source_as_of_date'], record_source=item['record_source'])
    for item in aircraft]

In [ ]:
# Public airline references; route, service, fleet, and flight participation remain synthetic.
airlines = load_catalog('airlines.json')
assert len(airlines) == num_airlines
assert len({airline['iata_code'] for airline in airlines}) == num_airlines
assert len({airline['icao_code'] for airline in airlines}) == num_airlines

airline_rows = [Row(
    airline_id=airline['airline_id'], iata=airline['iata_code'], icao=airline['icao_code'],
    iata_code=airline['iata_code'], icao_code=airline['icao_code'],
    airline_name=airline['airline_name'], home_country=airline['home_country'],
    home_country_code=airline['iso_country_code'], iso_country_code=airline['iso_country_code'],
    alliance=airline['alliance'], reference_status=airline['reference_status'],
    is_synthetic=False, data_classification='PublicReference',
    source_name=airline['source_name'], source_url=airline['source_url'],
    official_website=airline['official_website'], source_as_of_date=airline['source_as_of_date'],
    record_source=airline['record_source'])
    for airline in airlines]

In [ ]:
# Fictional organization, portfolio relationships, airline service, fleet eligibility, and runway assumptions.
hq_rows = [Row(
    headquarters_id='SYN-HQ-FR-001', group_id='SYN-AIRPORT-GROUP-001',
    headquarters_country='France', headquarters_city='SYN-FR-HEADQUARTERS',
    relationship_status='FictionalCaseStudyOnly')]
region_rows = [Row(
    operating_region_id=country['operating_region_id'],
    operating_region_name=country['country_name'], iso_country_code=country['iso_country_code'],
    parent_group_id='SYN-AIRPORT-GROUP-001') for country in countries]
ownership_rows = [Row(
    group_assignment_id=f"SYN-GROUP-ASSIGN-{airport['airport_id']}",
    group_id='SYN-AIRPORT-GROUP-001', headquarters_id='SYN-HQ-FR-001',
    operating_region_id=airport['operating_region_id'], airport_id=airport['airport_id'],
    relationship_type='FictionalDemonstrationRelationship',
    relationship_disclaimer='No ownership or operating claim; entirely synthetic demonstration relationship')
    for airport in airports]
runway_rows = [Row(
    runway_reference_id=f"SYN-RWY-{airport['airport_id']}-01", airport_id=airport['airport_id'],
    runway_label='Illustrative Synthetic Runway 01', representative_length_m=2800 + (index % 5) * 250,
    compatibility_category='SyntheticCodeE' if index % 3 == 0 else 'SyntheticCodeC',
    operational_use_prohibited=True)
    for index, airport in enumerate(airports)]

# Every fictional airline receives a plausible narrow/regional core plus a deterministic subset of wide-body types.
code_c_types = [item for item in aircraft if item['stand_category'] == 'C']
code_e_types = [item for item in aircraft if item['stand_category'] == 'E']
eligibility_rows = []
eligibility_pairs = set()
for airline_index, airline in enumerate(airlines):
    eligible_types = list(code_c_types)
    eligible_types.extend(
        aircraft_type for type_index, aircraft_type in enumerate(code_e_types)
        if (airline_index + type_index + random_seed) % 3 == 0)
    for aircraft_type in eligible_types:
        pair = (airline['airline_id'], aircraft_type['aircraft_type_id'])
        if pair not in eligibility_pairs:
            eligibility_pairs.add(pair)
            eligibility_rows.append(Row(
                eligibility_id=f"SYN-ELIG-{pair[0]}-{pair[1]}", airline_id=pair[0], aircraft_type_id=pair[1],
                eligibility_status='SyntheticEligible'))

service_rows = []
service_pairs = set()
for airport_index, airport in enumerate(airports):
    selected_airlines = [airline for airline_index, airline in enumerate(airlines) if (airport_index + airline_index + random_seed) % 3 != 0]
    for airline in selected_airlines:
        pair = (airport['airport_id'], airline['airline_id'])
        service_pairs.add(pair)
        service_rows.append(Row(
            service_id=f"SYN-SERVICE-{pair[0]}-{pair[1]}", airport_id=pair[0], airline_id=pair[1],
            service_status='SyntheticScheduledService'))

In [ ]:
# Gate and service-team dimensions.

gate_rows = []
gate_capacity_by_id = {}

for airport in airports:
    airport_id = airport['airport_id']

    for gate_number in range(1, gates_per_airport + 1):
        terminal = 'T1' if gate_number <= 3 else 'T2'

        if gate_number <= 3:
            gate_type = 'Contact'
            max_wingspan_m = 36.0
        elif gate_number <= 5:
            gate_type = 'Contact'
            max_wingspan_m = 65.0
        else:
            gate_type = 'Remote'
            max_wingspan_m = 65.0

        gate_id = f'{airport_id}-G{gate_number}'
        gate_capacity_by_id[gate_id] = max_wingspan_m

        gate_rows.append(Row(
            gate_id=gate_id,
            airport_id=airport_id,
            gate_code=f'G{gate_number}',
            terminal=terminal,
            gate_type=gate_type,
            max_wingspan_m=float(max_wingspan_m)
        ))

disciplines = ['Cleaning', 'Fueling', 'Catering', 'Baggage', 'Maintenance']
team_rows = []

for airport in airports:
    for discipline in disciplines:
        team_rows.append(Row(
            team_id=f"{airport['airport_id']}-{discipline[:3].upper()}",
            airport_id=airport['airport_id'],
            team_name=f"{discipline} Team {airport['iata_code']}",
            discipline=discipline,
            shift='Day'
        ))


In [ ]:
# Write effective configuration and synthetic master catalogs/assumptions.
config_rows = [Row(
    config_id='airport-ops-demo-v3', schema_version=demo_config['schema_version'],
    environment=demo_config['environment'], simulation_profile=selected_profile,
    processing_mode=PROCESSING_MODE, is_synthetic=True, random_seed=random_seed,
    base_date=base_date, observation_timestamp=BASE + timedelta(hours=sim_hours) - timedelta(minutes=1),
    reference_as_of_date=demo_config['reference_as_of_date'], generator_version=demo_config['generator_version'],
    scale_factor=int(demo_config['scale_factor']), operating_region_count=int(demo_config['operating_region_count']),
    corporate_headquarters_count=int(demo_config['corporate_headquarters_count']),
    airport_count=num_airports, airline_count=num_airlines, aircraft_type_count=num_aircraft_types,
    gates_per_airport=gates_per_airport, terminals_per_airport=int(demo_config['terminals_per_airport']),
    zones_per_terminal=int(demo_config['zones_per_terminal']), checkpoints_per_airport=int(demo_config['checkpoints_per_airport']),
    simulation_hours=sim_hours, flights_per_gate=flights_per_gate,
    queue_interval_minutes=queue_interval_minutes, asset_state_interval_hours=int(demo_config['asset_state_interval_hours']),
    routes_per_airport=int(demo_config['routes_per_airport']), employees_per_airport=int(demo_config['employees_per_airport']),
    passengers_per_flight_target=int(demo_config['passengers_per_flight_target']),
    retail_outlets_per_terminal=int(demo_config['retail_outlets_per_terminal']),
    maintenance_event_count=int(demo_config['maintenance_event_count']),
    incident_event_count=int(demo_config['incident_event_count']))]

write_delta(config_rows, 'bronze_demo_config')
write_delta(airport_rows, 'bronze_airport')
write_delta(country_rows, 'bronze_country')
write_delta(aircraft_rows, 'bronze_aircraft')
write_delta(airline_rows, 'bronze_airline')
write_delta(hq_rows, 'bronze_corporate_headquarters')
write_delta(region_rows, 'bronze_operating_region')
write_delta(ownership_rows, 'bronze_airport_group_assignment')
write_delta(runway_rows, 'bronze_runway_reference')
write_delta(eligibility_rows, 'bronze_airline_aircraft_eligibility')
write_delta(service_rows, 'bronze_airline_airport_service')
write_delta(gate_rows, 'bronze_gate')
write_delta(team_rows, 'bronze_service_team')

In [ ]:
# Synthetic scheduled, estimated, and actual flight/turnaround events with operational constraints.
flight_rows = []
flight_sequence = 0

for airport in airports:
    airport_id = airport['airport_id']
    airport_airline_ids = {airline_id for service_airport_id, airline_id in service_pairs if service_airport_id == airport_id}

    for gate_number in range(1, gates_per_airport + 1):
        gate_id = f'{airport_id}-G{gate_number}'
        gate_max_wingspan = gate_capacity_by_id[gate_id]
        feasible_pairs = [
            (airline, aircraft_type)
            for airline in airlines if airline['airline_id'] in airport_airline_ids
            for aircraft_type in aircraft
            if (airline['airline_id'], aircraft_type['aircraft_type_id']) in eligibility_pairs
            and float(aircraft_type['wingspan_m']) <= gate_max_wingspan]
        if not feasible_pairs:
            raise ValueError(f'No eligible airline/aircraft pair found for {gate_id}')

        for flight_offset in range(flights_per_gate):
            flight_sequence += 1
            selected_airline, selected_aircraft = random.choice(feasible_pairs)
            arrival_hour = (flight_offset * 5 + gate_number) % sim_hours
            day_index = arrival_hour // 24
            arrival_minute = random.choice([0, 10, 20, 30, 40, 50])
            turnaround_target = int(selected_aircraft['representative_turnaround_target_min'])
            scheduled_arrival = ts(arrival_hour, arrival_minute)
            scheduled_departure = scheduled_arrival + timedelta(minutes=turnaround_target + 20)
            delay_weights = [48, 20, 15, 11, 6] if day_index % 3 == 2 else [55, 20, 14, 8, 3]
            arrival_delay = random.choices([0, 5, 10, 20, 35], weights=delay_weights)[0]
            estimated_arrival = scheduled_arrival + timedelta(minutes=max(0, arrival_delay - 5))
            actual_arrival = scheduled_arrival + timedelta(minutes=arrival_delay)
            turnaround_start = actual_arrival + timedelta(minutes=5)
            turnaround_variance = random.choices([0, 5, 10, 15, 30], weights=[12, 28, 38, 17, 5])[0]
            turnaround_duration = max(20, turnaround_target + turnaround_variance)
            turnaround_end = turnaround_start + timedelta(minutes=turnaround_duration)
            estimated_departure = max(scheduled_departure, estimated_arrival + timedelta(minutes=turnaround_target + 5))
            departure_extra_delay = random.choices([0, 5, 10], weights=[70, 20, 10])[0]
            actual_departure = max(scheduled_departure, turnaround_end) + timedelta(minutes=departure_extra_delay)
            departure_delay = int((actual_departure - scheduled_departure).total_seconds() / 60)
            status = 'Delayed' if departure_delay > 15 else 'OnTime'
            delay_reason = random.choice(delay_reasons) if status == 'Delayed' else 'None'
            target_passengers = int(demo_config['passengers_per_flight_target']) * int(demo_config['scale_factor'])
            passenger_count = min(
                int(selected_aircraft['representative_seat_capacity']),
                max(1, int(target_passengers * random.uniform(0.75, 1.25))))

            flight_rows.append(Row(
                flight_event_id=f'SYN-FE-{flight_sequence:07d}', flight_no=f"{selected_airline['iata_code']}-{100 + flight_sequence}",
                airport_id=airport_id, gate_id=gate_id, airline_id=selected_airline['airline_id'],
                aircraft_type_id=selected_aircraft['aircraft_type_id'], scheduled_arrival=scheduled_arrival,
                estimated_arrival=estimated_arrival, actual_arrival=actual_arrival,
                scheduled_departure=scheduled_departure, estimated_departure=estimated_departure,
                actual_departure=actual_departure, turnaround_start=turnaround_start,
                turnaround_end=turnaround_end, passenger_count=passenger_count,
                status=status, delay_reason=delay_reason))

write_delta(flight_rows, 'bronze_flight_turnaround')

In [ ]:
# Synthetic passenger queues with recurring daily peaks and deterministic multi-day demand variation.
queue_rows = []
queue_sequence = 0
intervals = int(sim_hours * 60 / queue_interval_minutes)

for airport_index, airport in enumerate(airports):
    airport_id = airport['airport_id']
    for checkpoint_index, checkpoint in enumerate(checkpoints):
        for interval_number in range(intervals):
            queue_sequence += 1
            minute_offset = interval_number * queue_interval_minutes
            absolute_hour = int(minute_offset / 60)
            hour_of_day = absolute_hour % 24
            day_index = absolute_hour // 24
            peak_multiplier = 1.8 if hour_of_day in [6,7,8,17,18,19] else 1.0
            day_multiplier = 1.0 + 0.08 * ((day_index + airport_index) % 3)
            checkpoint_multiplier = [0.9,1.35,1.25,1.0][checkpoint_index]
            base_queue_length = random.randint(5, 25)
            queue_length = int(base_queue_length * peak_multiplier * day_multiplier * checkpoint_multiplier)
            wait_time = round(queue_length * random.uniform(0.4, 0.9), 1)
            throughput = int(queue_length * random.uniform(1.5, 3.0))
            forecast_factor = 0.96 + ((queue_sequence + random_seed) % 9) / 100.0
            queue_rows.append(Row(
                queue_metric_id=f'SYN-Q-{queue_sequence:09d}', airport_id=airport_id,
                checkpoint=checkpoint, event_time=ts(0, minute_offset),
                queue_length=queue_length, wait_time_min=float(wait_time),
                predicted_wait_time_min=round(float(wait_time) * forecast_factor, 1),
                throughput_pax=throughput))

write_delta(queue_rows, 'bronze_passenger_queue')

In [ ]:
# Synthetic energy metering with recurring daily peaks and multi-day demand variation.
energy_rows = []
energy_sequence = 0
base_kwh_map = {'HVAC': 40, 'Lighting': 12, 'Jetbridge': 8, 'GPU': 15}

for airport_index, airport in enumerate(airports):
    airport_id = airport['airport_id']
    for gate_number in range(1, gates_per_airport + 1):
        gate_id = f'{airport_id}-G{gate_number}'
        for hour in range(sim_hours):
            hour_of_day = hour % 24
            day_index = hour // 24
            day_multiplier = 1.0 + 0.04 * ((day_index + airport_index) % 3)
            for source in energy_sources:
                energy_sequence += 1
                load_multiplier = 1.5 if hour_of_day in [6,7,8,17,18,19] else 1.0
                kwh = round(base_kwh_map[source] * load_multiplier * day_multiplier * random.uniform(0.8, 1.2), 2)
                energy_rows.append(Row(
                    meter_reading_id=f'SYN-EN-{energy_sequence:010d}', airport_id=airport_id,
                    gate_id=gate_id, source=source, event_time=ts(hour), kwh=float(kwh)))

write_delta(energy_rows, 'bronze_energy')

In [ ]:
# Synthetic maintenance events parameterized by simulation profile.
asset_types = ['Jetbridge', 'BaggageBelt', 'HVAC', 'Escalator', 'Lighting']
severities = ['Low', 'Medium', 'High']
maintenance_teams_by_airport = {
    team.airport_id: team for team in team_rows if team.discipline == 'Maintenance'}
maintenance_event_count = int(demo_config['maintenance_event_count'])
maintenance_rows = []

for maintenance_number in range(1, maintenance_event_count + 1):
    airport = random.choice(airports)
    airport_id = airport['airport_id']
    gate_number = random.randint(1, gates_per_airport)
    maintenance_team = maintenance_teams_by_airport[airport_id]
    severity = random.choices(severities, weights=[55, 30, 15])[0]
    anomaly_flag = 1 if random.random() < 0.20 else 0
    hour = random.randint(0, sim_hours - 1)
    asset_type = random.choice(asset_types)
    maintenance_rows.append(Row(
        maintenance_id=f'SYN-MT-{maintenance_number:07d}', airport_id=airport_id,
        gate_id=f'{airport_id}-G{gate_number}', asset_type=asset_type,
        team_id=maintenance_team.team_id, event_time=ts(hour, random.randint(0, 59)),
        severity=severity, anomaly_flag=anomaly_flag,
        status=random.choice(['Open', 'InProgress', 'Closed']),
        description=f'Synthetic condition check on {asset_type}'))

write_delta(maintenance_rows, 'bronze_maintenance')

In [ ]:
# Climate-aware synthetic weather snapshots with daily cycles; not historical observations.
conditions = ['Clear', 'Cloudy', 'Windy', 'Rain', 'Fog']
temperature_ranges = {
    'France': (4.0, 29.0), 'Italy': (7.0, 34.0),
    'Portugal': (10.0, 31.0), 'Jordan': (8.0, 39.0)}
weather_rows = []
weather_sequence = 0

for airport_index, airport in enumerate(airports):
    airport_id = airport['airport_id']
    minimum_temperature, maximum_temperature = temperature_ranges[airport['country']]
    for hour in range(sim_hours):
        weather_sequence += 1
        hour_of_day = hour % 24
        day_index = hour // 24
        condition_weights = [50,24,14,8,4] if day_index % 3 != 2 else [35,28,17,15,5]
        condition = random.choices(conditions, weights=condition_weights)[0]
        visibility_km = random.uniform(1.0,4.0) if condition == 'Fog' else random.uniform(5.0,10.0)
        precipitation_mm = random.uniform(0.1,5.0) if condition == 'Rain' else 0.0
        cycle = abs(12 - hour_of_day) / 12.0
        temperature_center = maximum_temperature - (maximum_temperature-minimum_temperature)*0.55*cycle
        temperature = temperature_center + random.uniform(-2.0,2.0)
        weather_rows.append((
            f'SYN-WX-{weather_sequence:06d}', airport_id, ts(hour),
            float(round(max(minimum_temperature,min(maximum_temperature,temperature)),1)),
            float(round(random.uniform(0.0,45.0),1)), float(round(visibility_km,1)),
            float(round(precipitation_mm,1)), condition, 'SyntheticClimateAware'))

weather_schema = 'weather_id string, airport_id string, event_time timestamp, temperature_c double, wind_kph double, visibility_km double, precip_mm double, condition string, observation_type string'
write_delta(spark.createDataFrame(weather_rows, weather_schema), 'bronze_weather')

In [ ]:
# Synthetic operational, service, safety, security-simulation, and baggage incidents.
incident_categories = ['SafetySimulation', 'SecuritySimulation', 'Operational', 'Technical', 'Baggage', 'CustomerService']
incident_event_count = int(demo_config['incident_event_count'])
incident_rows = []

for incident_number in range(1, incident_event_count + 1):
    airport = random.choice(airports)
    airport_id = airport['airport_id']
    gate_number = random.randint(1, gates_per_airport)
    severity = random.choices(['Low', 'Medium', 'High'], weights=[55, 30, 15])[0]
    hour = random.randint(0, sim_hours - 1)
    category = random.choice(incident_categories)
    incident_rows.append((
        f'SYN-IN-{incident_number:07d}', airport_id, f'{airport_id}-G{gate_number}',
        ts(hour, random.randint(0,59)), category, severity,
        int(random.choices([0,5,15,30,60], weights=[40,25,20,10,5])[0]),
        random.choice(['Open','Resolved']), f'Synthetic {category} scenario at gate'))

incident_schema = 'incident_id string, airport_id string, gate_id string, event_time timestamp, category string, severity string, delay_minutes int, status string, description string'
write_delta(spark.createDataFrame(incident_rows, incident_schema), 'bronze_operational_incidents')

In [ ]:
# Bronze reference, metadata, volume, compatibility, and referential-integrity checks.
expected_counts = {
    'bronze_demo_config':1,'bronze_country':4,'bronze_airport':num_airports,
    'bronze_airline':num_airlines,'bronze_aircraft':num_aircraft_types,
    'bronze_corporate_headquarters':1,'bronze_operating_region':4,
    'bronze_airport_group_assignment':num_airports,'bronze_runway_reference':num_airports,
    'bronze_airline_aircraft_eligibility':len(eligibility_rows),
    'bronze_airline_airport_service':len(service_rows),
    'bronze_gate':num_airports*gates_per_airport,
    'bronze_service_team':num_airports*len(disciplines),
    'bronze_flight_turnaround':num_airports*gates_per_airport*flights_per_gate,
    'bronze_passenger_queue':num_airports*len(checkpoints)*(sim_hours*60//queue_interval_minutes),
    'bronze_energy':num_airports*gates_per_airport*sim_hours*len(energy_sources),
    'bronze_maintenance':maintenance_event_count,'bronze_weather':num_airports*sim_hours,
    'bronze_operational_incidents':incident_event_count}
for table_name, expected in expected_counts.items():
    actual=spark.table(table_name).count()
    assert actual==expected,f'{table_name}: {actual} != {expected}'

expected_region_counts={'France':6,'Italy':5,'Portugal':5,'Jordan':2}
airport_region_counts={row['country']:row['count'] for row in spark.table('bronze_airport').groupBy('country').count().collect()}
assert airport_region_counts==expected_region_counts
for table_name, code_columns, expected_count in [
    ('bronze_airport',['airport_id','iata_code','icao_code'],18),
    ('bronze_airline',['airline_id','iata_code','icao_code'],20),
    ('bronze_aircraft',['aircraft_type_id','icao_type_designator'],16)]:
    for column_name in code_columns:
        assert spark.table(table_name).select(column_name).distinct().count()==expected_count

required_metadata={'is_synthetic','data_classification','source_name','source_url','source_as_of_date','generated_at_utc','generator_version','random_seed','batch_id','record_source'}
public_reference_tables={'bronze_airport','bronze_country','bronze_airline','bronze_aircraft'}
valid_classifications={'PublicReference','SyntheticMaster','SyntheticOperational','DerivedAnalytical'}
for table_name in expected_counts:
    frame=spark.table(table_name)
    assert required_metadata.issubset(set(frame.columns)),table_name
    assert frame.filter(~F.col('data_classification').isin(sorted(valid_classifications))).count()==0,table_name
    if table_name in public_reference_tables:
        assert frame.filter(F.col('is_synthetic') | (F.col('data_classification')!='PublicReference')).count()==0,table_name
    else:
        assert frame.filter(~F.col('is_synthetic')).count()==0,table_name

incompatible_assignment_count=spark.sql("""
SELECT COUNT(*) incompatible_count FROM bronze_flight_turnaround f
JOIN bronze_gate g ON f.gate_id=g.gate_id
JOIN bronze_aircraft a ON f.aircraft_type_id=a.aircraft_type_id
WHERE a.wingspan_m>g.max_wingspan_m
""").first()['incompatible_count']
service_violation_count=spark.sql("""
SELECT COUNT(*) violations FROM bronze_flight_turnaround f
LEFT ANTI JOIN bronze_airline_airport_service s
ON f.airport_id=s.airport_id AND f.airline_id=s.airline_id
""").first()['violations']
eligibility_violation_count=spark.sql("""
SELECT COUNT(*) violations FROM bronze_flight_turnaround f
LEFT ANTI JOIN bronze_airline_aircraft_eligibility e
ON f.airline_id=e.airline_id AND f.aircraft_type_id=e.aircraft_type_id
""").first()['violations']
assert incompatible_assignment_count==0 and service_violation_count==0 and eligibility_violation_count==0
assert spark.table('bronze_airport_group_assignment').filter(~F.col('relationship_disclaimer').contains('fictional')).count()==0
print('PASS: Bronze 18-airport/20-airline/16-aircraft public references, synthetic bridges, metadata, volume, and compatibility contracts')